# 03 - Feature Engineering

Loads raw data from Athena, engineers features, and stores them in SageMaker Feature Store.

Features created:
- `review_length` — character count of review text
- `word_count` — word count of review text
- `vader_score` — VADER compound sentiment score (-1 to 1)
- `sentiment` — binary target: 0 = negative (1-2 stars), 1 = positive (4-5 stars)

Note: 3-star reviews are dropped as ambiguous.

**Prerequisite:** Run `01_setup_athena.ipynb` first to create `project_config.json`.

## 0. Install Dependencies

In [18]:
import importlib, subprocess

def install_if_missing(package):
    if importlib.util.find_spec(package) is None:
        print(f'Installing {package}...')
        subprocess.run(['pip', 'install', package, '--quiet'], check=True)
        print(f'{package} installed')
    else:
        print(f'{package} already installed, skipping')

install_if_missing('nltk')

import nltk
nltk.download('vader_lexicon', quiet=True)
print('Dependencies ready')

nltk already installed, skipping
Dependencies ready


## 1. Setup

In [19]:
import boto3, json, time
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer

with open('project_config.json') as f:
    cfg = json.load(f)

REGION         = cfg['REGION']
GLUE_DB        = cfg['GLUE_DB']
ATHENA_RESULTS = cfg['ATHENA_RESULTS']
SOURCE_BUCKET  = cfg['SOURCE_BUCKET']

session   = boto3.Session(region_name=REGION)
athena    = session.client('athena')
s3        = session.client('s3')
sm_client = session.client('sagemaker')

def get_role():
    try:
        import sagemaker
        return sagemaker.get_execution_role()
    except Exception:
        pass
    try:
        with open('/opt/ml/metadata/resource-metadata.json') as f:
            meta = json.load(f)
        return meta['ExecutionRoleArn']
    except Exception:
        pass
    try:
        iam = session.client('iam')
        return iam.get_role(RoleName='LabRole')['Role']['Arn']
    except Exception:
        pass
    raise RuntimeError('Could not detect IAM role — set MANUAL_ROLE_ARN manually')

MANUAL_ROLE_ARN = ''
role = MANUAL_ROLE_ARN if MANUAL_ROLE_ARN else get_role()

print(f'Region : {REGION}')
print(f'Role   : {role}')

Region : us-east-2
Role   : arn:aws:iam::203012547555:role/service-role/AmazonSageMaker-ExecutionRole-20260521T133544


## 2. Load Data from Athena

Samples 100k rows randomly from 2019 data.

In [20]:
def run_athena_query(sql):
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': GLUE_DB},
        ResultConfiguration={'OutputLocation': ATHENA_RESULTS}
    )
    query_id = response['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=query_id)
        state  = status['QueryExecution']['Status']['State']
        if state == 'SUCCEEDED':
            break
        elif state in ['FAILED', 'CANCELLED']:
            reason = status['QueryExecution']['Status']['StateChangeReason']
            raise Exception(f'Query {state}: {reason}')
        time.sleep(2)
    rows, next_token = [], None
    while True:
        kwargs = {'QueryExecutionId': query_id}
        if next_token:
            kwargs['NextToken'] = next_token
        page = athena.get_query_results(**kwargs)
        rows.extend(page['ResultSet']['Rows'])
        next_token = page.get('NextToken')
        if not next_token:
            break
    headers = [c['VarCharValue'] for c in rows[0]['Data']]
    data    = [[c.get('VarCharValue', '') for c in row['Data']] for row in rows[1:]]
    df = pd.DataFrame(data, columns=headers)
    # Fix dtypes — Athena returns everything as string
    df['stars']  = pd.to_numeric(df['stars'])
    df['useful'] = pd.to_numeric(df['useful'])
    df['funny']  = pd.to_numeric(df['funny'])
    df['cool']   = pd.to_numeric(df['cool'])
    df['date']   = pd.to_datetime(df['date'])
    return df

In [21]:
print('Loading data from Athena...')
df = run_athena_query(
    'SELECT * FROM yelp_reviews_2019 ORDER BY rand() LIMIT 100000'
)

print(f'Loaded {len(df):,} rows')
print(f'Star distribution:')
print(df['stars'].value_counts().sort_index())

Loading data from Athena...


Loaded 100,000 rows
Star distribution:
stars
1    17151
2     7217
3     8110
4    16307
5    51215
Name: count, dtype: int64


## 3. Create Binary Target

Drop 3-star reviews (ambiguous), then create binary sentiment label.

In [22]:
before = len(df)

df = df[df['stars'] != 3].copy()
df['sentiment'] = (df['stars'] >= 4).astype(int)

print(f'Dropped {before - len(df):,} three-star reviews')
print(f'Remaining rows: {len(df):,}')
print(f'\nClass distribution:')
print(df['sentiment'].value_counts())
print(f'\nPositive rate: {df["sentiment"].mean()*100:.1f}%')

Dropped 8,110 three-star reviews
Remaining rows: 91,890

Class distribution:
sentiment
1    67522
0    24368
Name: count, dtype: int64

Positive rate: 73.5%


## 4. Text Features

In [23]:
df['review_length'] = df['text'].str.len()
df['word_count']    = df['text'].str.split().str.len()

print('review_length stats:')
print(df['review_length'].describe())
print('\nword_count stats:')
print(df['word_count'].describe())

review_length stats:
count    91890.000000
mean       512.613712
std        476.864462
min         31.000000
25%        211.000000
50%        366.000000
75%        647.000000
max       5000.000000
Name: review_length, dtype: float64

word_count stats:
count    91890.000000
mean        94.679922
std         88.976824
min          1.000000
25%         38.000000
50%         67.000000
75%        120.000000
max       1050.000000
Name: word_count, dtype: float64


## 5. VADER Sentiment Score

Computes compound sentiment score (-1 = most negative, +1 = most positive).
This may take 2-3 minutes for 100k rows.

In [24]:
sid = SentimentIntensityAnalyzer()

print('Computing VADER scores...')
start = time.time()

df['vader_score'] = df['text'].apply(
    lambda x: sid.polarity_scores(x)['compound']
)

print(f'Done in {time.time() - start:.0f}s')
print(f'\nvader_score stats:')
print(df['vader_score'].describe())
print(f'\nAverage VADER score by sentiment:')
print(df.groupby('sentiment')['vader_score'].mean())

Computing VADER scores...


Done in 68s

vader_score stats:
count    91890.000000
mean         0.613185
std          0.593457
min         -0.999200
25%          0.612400
50%          0.913600
75%          0.968300
max          0.999800
Name: vader_score, dtype: float64

Average VADER score by sentiment:
sentiment
0   -0.105730
1    0.872635
Name: vader_score, dtype: float64


## 6. Feature Summary

In [25]:
feature_cols = ['review_length', 'word_count', 'useful', 'funny', 'cool', 'vader_score']
target_col   = 'sentiment'

print('Feature matrix shape:', df[feature_cols].shape)
print()
print('Features:')
print(df[feature_cols].describe())
print()
print('Target distribution:')
print(df[target_col].value_counts())

Feature matrix shape: (91890, 6)

Features:
       review_length    word_count        useful         funny         cool  \
count   91890.000000  91890.000000  91890.000000  91890.000000  91890.00000   
mean      512.613712     94.679922      0.939025      0.223833      0.47657   
std       476.864462     88.976824      2.854824      1.434985      2.42751   
min        31.000000      1.000000      0.000000      0.000000      0.00000   
25%       211.000000     38.000000      0.000000      0.000000      0.00000   
50%       366.000000     67.000000      0.000000      0.000000      0.00000   
75%       647.000000    120.000000      1.000000      0.000000      0.00000   
max      5000.000000   1050.000000    171.000000    107.000000    172.00000   

        vader_score  
count  91890.000000  
mean       0.613185  
std        0.593457  
min       -0.999200  
25%        0.612400  
50%        0.913600  
75%        0.968300  
max        0.999800  

Target distribution:
sentiment
1    67522
0  

## 7. Save Features to S3

In [26]:
df_features = df[[
    'review_id',
    'review_length',
    'word_count',
    'useful',
    'funny',
    'cool',
    'vader_score',
    'sentiment'
]].copy()

local_path = '/tmp/yelp_features.parquet'
s3_key     = 'features/yelp_features.parquet'

df_features.to_parquet(local_path, index=False)
s3.upload_file(local_path, SOURCE_BUCKET, s3_key)

print(f'Features saved to s3://{SOURCE_BUCKET}/{s3_key}')
print(f'Rows    : {len(df_features):,}')
print(f'Columns : {list(df_features.columns)}')

Features saved to s3://aai-540-group1-yelp-reviews/features/yelp_features.parquet
Rows    : 91,890
Columns : ['review_id', 'review_length', 'word_count', 'useful', 'funny', 'cool', 'vader_score', 'sentiment']


## 8. SageMaker Feature Store

In [27]:
FEATURE_GROUP_NAME = 'yelp-review-features'

try:
    sm_client.create_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierFeatureName='review_id',
        EventTimeFeatureName='event_time',
        FeatureDefinitions=[
            {'FeatureName': 'review_id',     'FeatureType': 'String'},
            {'FeatureName': 'review_length', 'FeatureType': 'Integral'},
            {'FeatureName': 'word_count',    'FeatureType': 'Integral'},
            {'FeatureName': 'useful',        'FeatureType': 'Integral'},
            {'FeatureName': 'funny',         'FeatureType': 'Integral'},
            {'FeatureName': 'cool',          'FeatureType': 'Integral'},
            {'FeatureName': 'vader_score',   'FeatureType': 'Fractional'},
            {'FeatureName': 'sentiment',     'FeatureType': 'Integral'},
            {'FeatureName': 'event_time',    'FeatureType': 'Fractional'},
        ],
        OnlineStoreConfig={'EnableOnlineStore': True},
        OfflineStoreConfig={
            'S3StorageConfig': {
                'S3Uri': f's3://{SOURCE_BUCKET}/feature-store/'
            }
        },
        RoleArn=role
    )
    print(f'Feature group {FEATURE_GROUP_NAME} created')
except sm_client.exceptions.ResourceInUse:
    print(f'Feature group {FEATURE_GROUP_NAME} already exists, skipping')

# Wait for ready
print('Waiting for feature group...')
while True:
    status = sm_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )['FeatureGroupStatus']
    if status == 'Created':
        print('Feature group ready')
        break
    elif status == 'CreateFailed':
        raise Exception('Feature group creation failed')
    time.sleep(5)

Feature group yelp-review-features already exists, skipping
Waiting for feature group...
Feature group ready


## 9. Update Config for Next Notebooks

In [28]:
cfg['FEATURE_GROUP_NAME'] = FEATURE_GROUP_NAME
cfg['FEATURES_S3_PATH']   = f's3://{SOURCE_BUCKET}/features/yelp_features.parquet'
cfg['FEATURE_COLS']       = feature_cols
cfg['TARGET_COL']         = target_col

with open('project_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('Config updated')
print(json.dumps(cfg, indent=2))

Config updated
{
  "REGION": "us-east-2",
  "SOURCE_BUCKET": "aai-540-group1-yelp-reviews",
  "GLUE_DB": "yelp_reviews_db",
  "ATHENA_RESULTS": "s3://aai-540-group1-yelp-reviews/athena-results/qmou/",
  "TABLES": [
    "yelp_reviews_2019",
    "yelp_reviews_2020_2022"
  ],
  "FEATURE_GROUP_NAME": "yelp-review-features",
  "FEATURES_S3_PATH": "s3://aai-540-group1-yelp-reviews/features/yelp_features.parquet",
  "FEATURE_COLS": [
    "review_length",
    "word_count",
    "useful",
    "funny",
    "cool",
    "vader_score"
  ],
  "TARGET_COL": "sentiment"
}
